In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

# =====================
# DEVICE + SEED
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)

Device: cuda


In [ ]:
# =====================
# DOMAIN
# =====================
x_min, x_max = 0.0, 2*np.pi
T_final = 0.3

In [ ]:
# =====================
# TRUE SOLUTION
# =====================
def true_solution(x, t):
    return torch.exp(-t)*torch.sin(x) + \
           torch.exp(-64*t)*torch.sin(2*x)

In [ ]:
# =====================
# NETWORK
# =====================
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=4):
        super().__init__()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.SiLU())

        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())

        layers.append(nn.Linear(hidden_dim, output_dim))

        self.model = nn.Sequential(*layers)
        self.init_weights()

    def init_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)

In [ ]:
# =====================
# TRAINING DATA
# =====================
N_r = 4000
N_ic = 512

x_r = torch.rand(N_r,1).to(device)*(x_max-x_min) + x_min
t_r = torch.rand(N_r,1).to(device)*T_final

x_ic = torch.rand(N_ic,1).to(device)*(x_max-x_min) + x_min
t_ic = torch.zeros_like(x_ic).to(device)

u_ic_true = true_solution(x_ic, t_ic).detach()

In [ ]:
# =====================
# PINN RESIDUAL (6TH ORDER)
# =====================
def pinn_residual(model, x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)

    u = model(torch.cat([x,t],dim=1))

    u_t = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    # 6 nested derivatives
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    u_xxx = torch.autograd.grad(u_xx, x, torch.ones_like(u_xx), create_graph=True)[0]
    u_xxxx = torch.autograd.grad(u_xxx, x, torch.ones_like(u_xxx), create_graph=True)[0]
    u_xxxxx = torch.autograd.grad(u_xxxx, x, torch.ones_like(u_xxxx), create_graph=True)[0]
    u_xxxxxx = torch.autograd.grad(u_xxxxx, x, torch.ones_like(u_xxxxx), create_graph=True)[0]

    return u_t + u_xxxxxx

In [ ]:
# =====================
# L2 VALIDATION
# =====================
def compute_l2_error(model):
    x = torch.linspace(x_min, x_max, 512).to(device)
    t_vals = torch.tensor([0.05,0.1,0.2]).to(device)

    X,T = torch.meshgrid(x,t_vals,indexing="ij")
    X_flat = X.reshape(-1,1)
    T_flat = T.reshape(-1,1)

    with torch.no_grad():
        u_pred = model(torch.cat([X_flat,T_flat],dim=1))
        u_true = true_solution(X_flat,T_flat)

    return torch.norm(u_pred-u_true)/torch.norm(u_true)

In [ ]:
# =====================
# TRAINING LOOP
# =====================
pinn = MLP(2,1).to(device)
optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)

epochs = 4000

torch.cuda.reset_peak_memory_stats()
start_total = time.time()

for epoch in range(epochs):

    optimizer.zero_grad()

    res = pinn_residual(pinn, x_r, t_r)
    loss_pde = torch.mean(res**2)

    u_ic_pred = pinn(torch.cat([x_ic,t_ic],dim=1))
    loss_ic = torch.mean((u_ic_pred - u_ic_true)**2)

    loss = loss_pde + loss_ic
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        l2 = compute_l2_error(pinn)
        print(f"Epoch {epoch}, Loss {loss.item():.3e}, L2 {l2.item():.3e}")

total_time = time.time() - start_total
memory_used = torch.cuda.max_memory_allocated()/(1024**2)

print("\nPINN Total Time:", total_time)
print("PINN Peak Memory (MB):", memory_used)

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:270.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0, Loss 9.505e-01, L2 9.070e-01
Epoch 200, Loss 2.919e-01, L2 7.026e-01
Epoch 400, Loss 2.529e-01, L2 5.209e-01
Epoch 600, Loss 2.354e-01, L2 5.475e-01
Epoch 800, Loss 1.334e-01, L2 7.269e-01
Epoch 1000, Loss 1.155e-01, L2 9.478e-01
Epoch 1200, Loss 3.465e-01, L2 1.038e+00
Epoch 1400, Loss 6.277e-02, L2 1.341e+00
Epoch 1600, Loss 7.397e-02, L2 1.134e+00
Epoch 1800, Loss 5.632e-02, L2 1.372e+00
Epoch 2000, Loss 5.244e-02, L2 1.390e+00
Epoch 2200, Loss 4.321e-02, L2 1.568e+00
Epoch 2400, Loss 4.318e-02, L2 1.518e+00
Epoch 2600, Loss 4.025e-02, L2 1.599e+00
Epoch 2800, Loss 3.999e-02, L2 1.621e+00
Epoch 3000, Loss 3.429e-02, L2 1.765e+00
Epoch 3200, Loss 3.103e-02, L2 1.882e+00
Epoch 3400, Loss 3.049e-02, L2 1.879e+00
Epoch 3600, Loss 3.219e-02, L2 1.823e+00
Epoch 3800, Loss 3.034e-02, L2 1.851e+00

PINN Total Time: 2495.7964844703674
PINN Peak Memory (MB): 3461.91455078125


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

# =====================
# DEVICE + SEED
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)

# =====================
# DOMAIN
# =====================
x_min, x_max = 0.0, 2*np.pi
T_final = 0.3

# =====================
# TRUE SOLUTION
# =====================
def true_solution(x, t):
    return torch.exp(-t)*torch.sin(x) + \
           torch.exp(-64*t)*torch.sin(2*x)

# =====================
# NETWORK
# =====================
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=128, num_layers=4):
        super().__init__()

        layers = []
        layers.append(nn.Linear(input_dim, hidden_dim))
        layers.append(nn.SiLU())

        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.SiLU())

        layers.append(nn.Linear(hidden_dim, output_dim))

        self.model = nn.Sequential(*layers)
        self.init_weights()

    def init_weights(self):
        for m in self.model:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.model(x)

# =====================
# FOURIER MODES
# =====================
K = 8
k_vals = torch.arange(-K, K+1).float().to(device)
num_modes = len(k_vals)

# =====================
# TRAINING TIME SAMPLES
# =====================
N_t = 1000
t_train = torch.rand(N_t,1).to(device)*T_final

# Build vectorized batch
t_repeat = t_train.repeat(num_modes,1)
k_repeat = k_vals.repeat_interleave(N_t).reshape(-1,1)

# =====================
# INITIAL FOURIER COEFFS
# =====================
# sin(x) + sin(2x)
# sin(kx) = (e^{ikx} - e^{-ikx})/(2i)

def true_fourier_ic(k_vals):
    real = torch.zeros_like(k_vals)
    imag = torch.zeros_like(k_vals)

    imag[k_vals==1] = -0.5
    imag[k_vals==-1] = 0.5
    imag[k_vals==2] = -0.5
    imag[k_vals==-2] = 0.5

    return torch.stack([real, imag], dim=1)

# =====================
# SINN MODEL
# =====================
sinn = MLP(2,2).to(device)
optimizer = torch.optim.Adam(sinn.parameters(), lr=1e-3)

# =====================
# RESIDUAL (6TH ORDER)
# =====================
def sinn_residual(model, k, t):
    t.requires_grad_(True)

    inputs = torch.cat([k,t],dim=1)
    u_hat = model(inputs)

    u_real = u_hat[:,0:1]
    u_imag = u_hat[:,1:2]

    u_real_t = torch.autograd.grad(
        u_real, t,
        grad_outputs=torch.ones_like(u_real),
        create_graph=True
    )[0]

    u_imag_t = torch.autograd.grad(
        u_imag, t,
        grad_outputs=torch.ones_like(u_imag),
        create_graph=True
    )[0]

    k6 = k**6

    res_real = u_real_t + k6*u_real
    res_imag = u_imag_t + k6*u_imag

    return res_real, res_imag

# =====================
# L2 VALIDATION
# =====================
def compute_l2_error(model):
    x = torch.linspace(x_min, x_max, 512).to(device)
    t_vals = torch.tensor([0.05,0.1,0.2]).to(device)

    X,T = torch.meshgrid(x,t_vals,indexing="ij")
    X_flat = X.reshape(-1,1)
    T_flat = T.reshape(-1,1)

    u_pred = torch.zeros_like(X_flat)

    with torch.no_grad():
        for k in k_vals:
            k_tensor = torch.ones_like(T_flat)*k
            coeff = model(torch.cat([k_tensor,T_flat],dim=1))

            real = coeff[:,0:1]
            imag = coeff[:,1:2]

            u_pred += real*torch.cos(k*X_flat) - \
                      imag*torch.sin(k*X_flat)

    u_true = true_solution(X_flat,T_flat)

    return torch.norm(u_pred-u_true)/torch.norm(u_true)

# =====================
# TRAINING LOOP
# =====================
epochs = 4000

torch.cuda.reset_peak_memory_stats()
start_total = time.time()

for epoch in range(epochs):

    optimizer.zero_grad()

    res_real, res_imag = sinn_residual(sinn, k_repeat, t_repeat)

    # scale loss for conditioning
    loss_pde = (torch.mean(res_real**2) + torch.mean(res_imag**2)) / (K**12)

    # IC loss
    k_ic = k_vals.reshape(-1,1)
    t_ic = torch.zeros_like(k_ic)

    pred_ic = sinn(torch.cat([k_ic,t_ic],dim=1))
    true_ic = true_fourier_ic(k_vals).to(device)

    loss_ic = torch.mean((pred_ic - true_ic)**2)

    loss = loss_pde + loss_ic
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        l2 = compute_l2_error(sinn)
        print(f"Epoch {epoch}, Loss {loss.item():.3e}, L2 {l2.item():.3e}")

total_time = time.time() - start_total
memory_used = torch.cuda.max_memory_allocated()/(1024**2)

print("\nSINN Total Time:", total_time)
print("SINN Peak Memory (MB):", memory_used)


Device: cuda
Epoch 0, Loss 4.188e-02, L2 1.032e+00
Epoch 200, Loss 1.439e-03, L2 9.869e-01
Epoch 400, Loss 1.204e-03, L2 1.039e+00
Epoch 600, Loss 8.579e-04, L2 1.072e+00
Epoch 800, Loss 6.533e-04, L2 1.111e+00
Epoch 1000, Loss 5.119e-04, L2 1.149e+00
Epoch 1200, Loss 3.073e-04, L2 1.180e+00
Epoch 1400, Loss 2.127e-05, L2 1.224e+00
Epoch 1600, Loss 3.087e-06, L2 1.233e+00
Epoch 1800, Loss 1.327e-06, L2 1.234e+00
Epoch 2000, Loss 5.294e-07, L2 1.234e+00
Epoch 2200, Loss 4.237e-07, L2 1.234e+00
Epoch 2400, Loss 4.488e-07, L2 1.235e+00
Epoch 2600, Loss 2.727e-07, L2 1.235e+00
Epoch 2800, Loss 2.501e-07, L2 1.235e+00
Epoch 3000, Loss 4.602e-07, L2 1.235e+00
Epoch 3200, Loss 2.182e-05, L2 1.204e+00
Epoch 3400, Loss 1.313e-07, L2 1.235e+00
Epoch 3600, Loss 1.567e-07, L2 1.236e+00
Epoch 3800, Loss 1.387e-07, L2 1.237e+00

SINN Total Time: 84.71404075622559
SINN Peak Memory (MB): 500.84814453125
